<a href="https://colab.research.google.com/github/DimiGretsistas/Automated-Customers-Reviews-Project/blob/main/notebooks/04_summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Review Summarization

**Goals**
- Use a small generative model to create short recommendation articles from clustered product reviews.
- This notebook recreates the clustered sample directly, so it does not depend on the clustering notebook.

In [2]:
import pandas as pd

#load cleaned dataset from Colab
df = pd.read_csv(
    "/content/clean_reviews.csv",
    engine="python",
    on_bad_lines="skip"
)

# inspect data
df.head()

,name,reviews.rating,reviews.text,sentiment,clean_review
0,AmazonBasics AAA Performance Alkaline Batterie...,3,I order 3 of them and one of the item is bad q...,neutral,i order 3 of them and one of the item is bad q...
1,AmazonBasics AAA Performance Alkaline Batterie...,4,Bulk is always the less expensive way to go fo...,positive,bulk is always the less expensive way to go fo...
2,AmazonBasics AAA Performance Alkaline Batterie...,5,Well they are not Duracell but for the price i...,positive,well they are not duracell but for the price i...
3,AmazonBasics AAA Performance Alkaline Batterie...,5,Seem to work as well as name brand batteries a...,positive,seem to work as well as name brand batteries a...
4,AmazonBasics AAA Performance Alkaline Batterie...,5,These batteries are very long lasting the pric...,positive,these batteries are very long lasting the pric...


## Create Sample for Summarization
- Using a smaller sample to keep the notebook fast.

In [3]:
#use a smaller sample for fast processing
sample_df = df.sample(n=1000, random_state=42).copy()

#check sample size
sample_df.shape

(1000, 5)

## Create Embeddings and Clusters
- Using a pretrained sentence transformer and KMeans to recreate product clusters.

In [4]:
!pip install sentence-transformers -q

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

#prepare review texts
texts = sample_df["clean_review"].tolist()

#load embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

#create embeddings
embeddings = embedding_model.encode(texts, show_progress_bar=True)

#create 5 clusters
kmeans = KMeans(n_clusters=5, random_state=42)

#assign clusters
sample_df["cluster"] = kmeans.fit_predict(embeddings)

sample_df[["name", "clean_review", "cluster"]].head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

,name,clean_review,cluster
31722,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",this is a great tablet at a great price it is ...,2
23643,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16...",i bought this for my 6yr old am impressed w th...,2
12086,"Kindle Voyage E-reader, 6 High-Resolution Disp...",i like the small light weight size and the ada...,0
10989,Amazon Kindle Charger Power Adapter Wall Charg...,this was exactly what i needed knew i would fi...,4
25085,"Fire HD 8 Tablet with Alexa, 8 HD Display, 16 ...",i love my amazon fire i purchased one for my 1...,0


## Inspect Clustered Sample
- Checking available clusters and products in the sampled data.

In [5]:
#check sample dataset size
print(sample_df.shape)

#check available clusters
print(sample_df["cluster"].value_counts())

#inspect clustered sample
sample_df[["name", "reviews.rating", "sentiment", "cluster"]].head()

(1000, 6)
cluster
2    222
0    218
4    218
1    181
3    161
Name: count, dtype: int64


,name,reviews.rating,sentiment,cluster
31722,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",5,positive,2
23643,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16...",4,positive,2
12086,"Kindle Voyage E-reader, 6 High-Resolution Disp...",4,positive,0
10989,Amazon Kindle Charger Power Adapter Wall Charg...,5,positive,4
25085,"Fire HD 8 Tablet with Alexa, 8 HD Display, 16 ...",5,positive,0


## Prepare Product Insights
- Extracting top products, worst product, and complaints per cluster.

In [6]:
def prepare_cluster_insights(cluster_df):
    #calculate average rating per product
    product_stats = cluster_df.groupby("name").agg(
        avg_rating=("reviews.rating", "mean"),
        review_count=("reviews.rating", "count")
    ).reset_index()

    #keep products with at least 3 reviews
    product_stats = product_stats[product_stats["review_count"] >= 3]

    #top 3 products
    top_products = product_stats.sort_values(
        by="avg_rating",
        ascending=False
    ).head(3)

    #worst product
    worst_product = product_stats.sort_values(
        by="avg_rating",
        ascending=True
    ).head(1)

    #complaints (negative reviews)
    complaints = cluster_df[
        cluster_df["sentiment"] == "negative"
    ]["clean_review"].head(5).tolist()

    return top_products, worst_product, complaints

In [7]:
#test for one cluster
cluster_df = sample_df[sample_df["cluster"] == 0]

top_products, worst_product, complaints = prepare_cluster_insights(cluster_df)

print("Top products:")
print(top_products)

print("\nWorst product:")
print(worst_product)

print("\nComplaints:")
print(complaints)

Top products:
                                                 name  avg_rating  \
18  Fire HD 8 Tablet with Alexa, 8 HD Display, 16 ...    4.787879   
13  Amazon Fire HD 8 with Alexa (8" HD Display Tab...    4.666667   
19  Fire HD 8 Tablet with Alexa, 8 HD Display, 32 ...    4.666667   

    review_count  
18            33  
13             3  
19             3  

Worst product:
                                                 name  avg_rating  \
22  Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16...    3.833333   

    review_count  
22             6  

Complaints:
['1st kindle screen failed had to reboot often customer service couldn t fix replaced 2nd one screen reboots often customer service offered replace just want one that works can t get upgrade replacement', 'returned it it was a poor reader and clearly would not last', 'i bought this because i have ebooks in college not only could i not get my ebook i called the school it department they could not either my semester was alrea

##Clean Product Names
- Fixing formatting issues in product names.

In [8]:
#clean messy product names (remove commas, line breaks)
sample_df["name"] = sample_df["name"].str.replace(r"[\r\n,]+", " ", regex=True).str.strip()

##-----------------------------------------------------------
## Inspect Clustered Sample
- Checking available clusters and products in the sampled data.

In [9]:
#check sample dataset size
print(sample_df.shape)

#check available clusters
print(sample_df["cluster"].value_counts())

#inspect clustered sample
sample_df[["name", "reviews.rating", "sentiment", "cluster"]].head()


(1000, 6)
cluster
2    222
0    218
4    218
1    181
3    161
Name: count, dtype: int64


,name,reviews.rating,sentiment,cluster
31722,Fire Tablet 7 Display Wi-Fi 8 GB - Includes...,5,positive,2
23643,Fire Kids Edition Tablet 7 Display Wi-Fi 16...,4,positive,2
12086,Kindle Voyage E-reader 6 High-Resolution Disp...,4,positive,0
10989,Amazon Kindle Charger Power Adapter Wall Charg...,5,positive,4
25085,Fire HD 8 Tablet with Alexa 8 HD Display 16 ...,5,positive,0


##Load Generator Model
- Using a small generative model to create recommendation text.

In [10]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    max_new_tokens=300,
    device_map="auto"
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


##Generate Recommendation Article
- Create a short blog-style summary for one cluster.

In [11]:
#select cluster
cluster_id = 0

cluster_df = sample_df[sample_df["cluster"] == cluster_id]

top_products, worst_product, complaints = prepare_cluster_insights(cluster_df)

prompt = f"""
Write a short product recommendation article.

Top products:
{top_products.to_string(index=False)}

Worst product:
{worst_product.to_string(index=False)}

Complaints:
{complaints}

Include:
- Top 3 products
- Differences
- Complaints
- Worst product warning
"""

result = generator(prompt)

print(result[0]["generated_text"])

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Write a short product recommendation article.

Top products:
                                                                             name  avg_rating  review_count
Fire HD 8 Tablet with Alexa  8 HD Display  16 GB  Tangerine - with Special Offers    4.787879            33
                               Amazon Fire HD 8 with Alexa (8" HD Display Tablet)    4.666667             3
Fire HD 8 Tablet with Alexa  8 HD Display  32 GB  Tangerine - with Special Offers    4.666667             3

Worst product:
                                                                  name  avg_rating  review_count
Fire Kids Edition Tablet  7 Display  Wi-Fi  16 GB  Pink Kid-Proof Case    3.833333             6

Complaints:
['1st kindle screen failed had to reboot often customer service couldn t fix replaced 2nd one screen reboots often customer service offered replace just want one that works can t get upgrade replacement', 'returned it it was a poor reader and clearly would not last', 'i bought this 

In [12]:
#Prompt Variation 2

prompt = f"""
You are a tech reviewer.

Write a short, clear product recommendation article based ONLY on the data below.

Top products:
{top_products.to_string(index=False)}

Worst product:
{worst_product.to_string(index=False)}

Customer complaints:
{complaints}

Rules:
- Be concise and factual
- Do NOT invent products
- Do NOT mention brands not listed
- Focus on differences and real issues

Structure:
1. Introduction
2. Top products comparison
3. Key complaints
4. Worst product warning
5. Final recommendation
"""

result = generator(prompt)

print(result[0]["generated_text"])

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a tech reviewer.

Write a short, clear product recommendation article based ONLY on the data below.

Top products:
                                                                             name  avg_rating  review_count
Fire HD 8 Tablet with Alexa  8 HD Display  16 GB  Tangerine - with Special Offers    4.787879            33
                               Amazon Fire HD 8 with Alexa (8" HD Display Tablet)    4.666667             3
Fire HD 8 Tablet with Alexa  8 HD Display  32 GB  Tangerine - with Special Offers    4.666667             3

Worst product:
                                                                  name  avg_rating  review_count
Fire Kids Edition Tablet  7 Display  Wi-Fi  16 GB  Pink Kid-Proof Case    3.833333             6

Customer complaints:
['1st kindle screen failed had to reboot often customer service couldn t fix replaced 2nd one screen reboots often customer service offered replace just want one that works can t get upgrade replacement', 'return

In [13]:
#Prompt variation 3

prompt = f"""
You are a tech reviewer.

Write a short, structured product recommendation article based ONLY on the data below.

Top products:
{top_products.to_string(index=False)}

Worst product:
{worst_product.to_string(index=False)}

Customer complaints:
{complaints}

Rules:
- Only use the information provided
- Do NOT invent features or products
- Be concise and factual

Structure:
1. Introduction
2. Top products comparison (bullet points)
3. Key complaints (bullet points)
4. Worst product warning
5. Final recommendation
"""

result = generator(prompt)

print(result[0]["generated_text"])

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a tech reviewer.

Write a short, structured product recommendation article based ONLY on the data below.

Top products:
                                                                             name  avg_rating  review_count
Fire HD 8 Tablet with Alexa  8 HD Display  16 GB  Tangerine - with Special Offers    4.787879            33
                               Amazon Fire HD 8 with Alexa (8" HD Display Tablet)    4.666667             3
Fire HD 8 Tablet with Alexa  8 HD Display  32 GB  Tangerine - with Special Offers    4.666667             3

Worst product:
                                                                  name  avg_rating  review_count
Fire Kids Edition Tablet  7 Display  Wi-Fi  16 GB  Pink Kid-Proof Case    3.833333             6

Customer complaints:
['1st kindle screen failed had to reboot often customer service couldn t fix replaced 2nd one screen reboots often customer service offered replace just want one that works can t get upgrade replacement', 'r

##Final Conclusions

- The project successfully demonstrates an end-to-end NLP pipeline for analyzing customer reviews.

- The sentiment classifier (DistilBERT) achieved high performance (~95% accuracy), but results are influenced by dataset imbalance, leading to weaker detection of neutral reviews.

- The clustering approach (MiniLM embeddings + KMeans) revealed meaningful product groupings, although quantitative evaluation (low silhouette score) shows that text clustering remains challenging.

- The summarization model (Qwen 1.5B) was able to generate structured recommendation articles using extracted insights such as top products and customer complaints.

- Prompt design proved to be critical for controlling output quality and reducing hallucinations in generative models.

- Overall, combining classification, clustering, and generation provides valuable business insights, transforming raw reviews into actionable recommendations.

- Limitations include dataset imbalance, noisy text data, and the inherent challenges of unsupervised clustering and generative AI reliability.

## N-Shot Prompting

- Instead of fine-tuning the generator today, we improved the model output using n-shot prompting.  
- This means we provide the model with examples of the expected output format before asking it to generate a new product recommendation article.

In [18]:
n_shot_prompt = f"""
You are a tech product reviewer.

Example:

Input:
Top products: Product A, Product B, Product C
Complaints: battery issues, slow setup
Worst product: Product D

Output:
Product A performs best overall based on customer ratings.
Product B offers a good balance between features and usability.
Product C is suitable for basic use cases.

Common complaints include battery problems and setup difficulties.
Product D should be avoided due to negative customer feedback.

---

Now use ONLY the real data below.

Top products:
{top_products.to_string(index=False)}

Worst product:
{worst_product.to_string(index=False)}

Customer complaints:
{complaints}

Rules:
- ONLY use the products listed above
- DO NOT reuse example names
- Be concise and factual
- Mention top products, complaints, worst product, and recommendation
"""

result = generator(n_shot_prompt)

print(result[0]["generated_text"])

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a tech product reviewer.

Example:

Input:
Top products: Product A, Product B, Product C
Complaints: battery issues, slow setup
Worst product: Product D

Output:
Product A performs best overall based on customer ratings.
Product B offers a good balance between features and usability.
Product C is suitable for basic use cases.

Common complaints include battery problems and setup difficulties.
Product D should be avoided due to negative customer feedback.

---

Now use ONLY the real data below.

Top products:
                                                                             name  avg_rating  review_count
Fire HD 8 Tablet with Alexa  8 HD Display  16 GB  Tangerine - with Special Offers    4.787879            33
                               Amazon Fire HD 8 with Alexa (8" HD Display Tablet)    4.666667             3
Fire HD 8 Tablet with Alexa  8 HD Display  32 GB  Tangerine - with Special Offers    4.666667             3

Worst product:
                             

##N-Shot Prompting Insights

- N-shot prompting improved the structure of the generated recommendation article.
- The model followed the example format and used the real product data more effectively.
- Some minor inaccuracies can still appear, showing that generated outputs require validation.
- This confirms that prompt design strongly affects summarization quality.